<div style="background-color: #90EE90; padding: 20px; border-radius: 10px;">
### House grouping system
We want to be able to classify houses according to their region and median income. To do this, we will use the famous California Housing dataset. It was constructed using data from the 1990 California census. It contains one row per census block group. A block group is the smallest geographic unit for which US Census data is published.
</div>

In [1]:
#CONFIGURACIÓN DEL ENTORNO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Algoritmos Naive Bayes
from sklearn.naive_bayes import GaussianNB

# Librerías para preprocesamiento y Machine Learning 
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import (
    classification_report, 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import pandas as pd
import numpy as np




from pickle import dump
from pathlib import Path

# Suprimir advertencias
import warnings
warnings.filterwarnings('ignore')

# Establecer estilo de visualización
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ ¡Librerías importadas exitosamente!")


✓ ¡Librerías importadas exitosamente!


In [2]:
# Load preprocessed data
df = pd.read_csv('../data/raw/housing.csv')
print(df.head())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  


<div style="background-color: #90EE90; padding: 20px; border-radius: 10px;">

### Step 1: Loading the dataset
Or download it and add it by hand in your repository. In this case, we are only interested in the Latitude, Longitude and MedInc columns.
</div>

In [3]:
df_processed = df.drop(['HouseAge','AveRooms','AveBedrms','Population','AveOccup','MedHouseVal'], axis=1)
print(df_processed.head())

   MedInc  Latitude  Longitude
0  8.3252     37.88    -122.23
1  8.3014     37.86    -122.22
2  7.2574     37.85    -122.24
3  5.6431     37.85    -122.25
4  3.8462     37.85    -122.25


<div style="background-color: #90EE90; padding: 20px; border-radius: 10px;">

### Step 2: Build a K-Means
Classify the data into 6 clusters using the K-Means model. Then store the cluster to which each house belongs as a new column in the dataset. You could call it cluster. To introduce it to your dataset, you may have to categorize it. See what format and values it has, and act accordingly. Plot it in a dot plot and describe what you see.
</div>

In [4]:
# Build K-Means model with 6 clusters
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)

# Fit the model and predict clusters
clusters_kmeans = kmeans.fit_predict(df_processed)

# Check the format and values
print("Cluster labels:", np.unique(clusters_kmeans))
print("Cluster label type:", type(clusters_kmeans[0]))
print("Shape:", clusters_kmeans.shape)

# Add cluster column to dataset
df_processed['cluster'] = clusters_kmeans

# Display first few rows to see the new column
print("\n" + "="*70)
print("DATASET WITH CLUSTER COLUMN")
print("="*70)
print(df_processed.head(5))

# Check cluster distribution
print("\n" + "="*70)
print("CLUSTER DISTRIBUTION")
print("="*70)
cluster_counts = df_processed['cluster'].value_counts().sort_index()
print(cluster_counts)
print(f"\nTotal houses: {len(df_processed)}")

Cluster labels: [0 1 2 3 4 5]
Cluster label type: <class 'numpy.int32'>
Shape: (20640,)

DATASET WITH CLUSTER COLUMN
   MedInc  Latitude  Longitude  cluster
0  8.3252     37.88    -122.23        5
1  8.3014     37.86    -122.22        5
2  7.2574     37.85    -122.24        5
3  5.6431     37.85    -122.25        5
4  3.8462     37.85    -122.25        0

CLUSTER DISTRIBUTION
cluster
0    4849
1    7011
2     459
3    3840
4    1683
5    2798
Name: count, dtype: int64

Total houses: 20640


<div style="background-color: #90EE90; padding: 20px; border-radius: 10px;">

### Step 3: Predict with the test set
Now use the trained model with the test set and add the points to the above plot to confirm that the prediction is successful or not.

</div>

In [5]:
# Step 3: Split data into train and test sets
from sklearn.model_selection import train_test_split

# Remove the 'cluster' column to get back to original features
df_features = df_processed.drop('cluster', axis=1)

# Split the data: 80% training, 20% testing
X_train, X_test, idx_train, idx_test = train_test_split(
    df_features, 
    df_features.index, 
    test_size=0.2, 
    random_state=42
)

print("="*70)
print("DATA SPLIT")
print("="*70)
print(f"Training set size: {len(X_train)} houses ({len(X_train)/len(df_features)*100:.1f}%)")
print(f"Test set size: {len(X_test)} houses ({len(X_test)/len(df_features)*100:.1f}%)")
print(f"Total: {len(df_features)} houses")

# Train K-Means on training data only
kmeans_train = KMeans(n_clusters=6, random_state=42, n_init=10)
train_clusters = kmeans_train.fit_predict(X_train)

# Predict clusters for test data
test_clusters = kmeans_train.predict(X_test)

print("\n" + "="*70)
print("CLUSTER DISTRIBUTION")
print("="*70)
print("\nTraining Set:")
train_counts = pd.Series(train_clusters).value_counts().sort_index()
print(train_counts)

print("\nTest Set:")
test_counts = pd.Series(test_clusters).value_counts().sort_index()
print(test_counts)

# Add cluster assignments to dataframes
X_train_with_clusters = X_train.copy()
X_train_with_clusters['cluster'] = train_clusters
X_train_with_clusters['set'] = 'train'

X_test_with_clusters = X_test.copy()
X_test_with_clusters['cluster'] = test_clusters
X_test_with_clusters['set'] = 'test'

# Combine for easier plotting
df_combined = pd.concat([X_train_with_clusters, X_test_with_clusters])

print("\nModel trained on training set and predictions made on test set!")

DATA SPLIT
Training set size: 16512 houses (80.0%)
Test set size: 4128 houses (20.0%)
Total: 20640 houses

CLUSTER DISTRIBUTION

Training Set:
0    1348
1     342
2    5628
3    3024
4    3919
5    2251
Name: count, dtype: int64

Test Set:
0     338
1      89
2    1488
3     736
4     957
5     520
Name: count, dtype: int64

Model trained on training set and predictions made on test set!


<div style="background-color: #90EE90; padding: 20px; border-radius: 10px;">

### Step 4: Train Supervised Classification Models
Now train supervised models using cluster labels as the target variable. We'll compare:
- **Logistic Regression** 
- **LinearSVC** 
- **Random Forest** 
- **GaussianNB** 
- **RandomForest** 
</div>

In [6]:
print("="*90)
print("STEP 4: KMEANS -> CLASIFICACIÓN SUPERVISADA".center(90))
print("="*90)

# 1) Usamos las pseudo-etiquetas de KMeans del Step 3
kmeans_train_labels = train_clusters   # from kmeans_train.fit_predict(X_train)
kmeans_test_labels  = test_clusters    # from kmeans_train.predict(X_test)

print(f"Pseudo-etiquetas (6 clusters de KMeans):")
print(f"  Train: {np.bincount(kmeans_train_labels)}")
print(f"  Test : {np.bincount(kmeans_test_labels)}")

# 2) Modelos supervisados para aprender las pseudo-etiquetas
# NOTE: GaussianNB is used instead of MultinomialNB because the housing
# features (Longitude, Latitude, MedInc) contain negative values.
# MultinomialNB only accepts non-negative inputs (designed for count data).
# GaussianNB assumes a normal distribution and works with any real-valued features.
candidate_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    "LinearSVC": LinearSVC(random_state=42),
    "GaussianNB": GaussianNB(),
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=15, random_state=42, n_jobs=-1
    ),
}

results_step4 = []

for name, clf in candidate_models.items():
    clf.fit(X_train, kmeans_train_labels)
    pred_train = clf.predict(X_train)
    pred_test  = clf.predict(X_test)

    results_step4.append({
        "Modelo": name,
        "Acc Train": accuracy_score(kmeans_train_labels, pred_train),
        "Acc Test":  accuracy_score(kmeans_test_labels, pred_test),
        "Precision Test": precision_score(kmeans_test_labels, pred_test, average='weighted', zero_division=0),
        "Recall Test":    recall_score(kmeans_test_labels, pred_test, average='weighted', zero_division=0),
        "F1 Test":        f1_score(kmeans_test_labels, pred_test, average='weighted', zero_division=0),
    })

results_step4_df = (pd.DataFrame(results_step4)
                      .sort_values("F1 Test", ascending=False)
                      .reset_index(drop=True))

print("\nResultados (supervisado aprendiendo pseudo-etiquetas de KMeans):")
display(results_step4_df.style.format({
    "Acc Train": "{:.4f}",
    "Acc Test":  "{:.4f}",
    "Precision Test": "{:.4f}",
    "Recall Test":    "{:.4f}",
    "F1 Test":        "{:.4f}",
}).highlight_max(color="lightgreen", axis=0, subset=["Acc Test", "F1 Test"]))

best_name = results_step4_df.loc[0, "Modelo"]
best_clf  = candidate_models[best_name]
best_pred = best_clf.predict(X_test)

print(f"\nMejor modelo: {best_name}")
print("\nClassification report (supervised vs KMeans labels en test):")
print(classification_report(kmeans_test_labels, best_pred,
                            target_names=[f'Cluster {i}' for i in range(6)],
                            digits=4))



                       STEP 4: KMEANS -> CLASIFICACIÓN SUPERVISADA                        
Pseudo-etiquetas (6 clusters de KMeans):
  Train: [1348  342 5628 3024 3919 2251]
  Test : [ 338   89 1488  736  957  520]

Resultados (supervisado aprendiendo pseudo-etiquetas de KMeans):


,Modelo,Acc Train,Acc Test,Precision Test,Recall Test,F1 Test
0,RandomForest,1.0000,0.9961,0.9961,0.9961,0.9961
1,GaussianNB,0.9623,0.9629,0.9632,0.9629,0.9628
2,LogisticRegression,0.9185,0.9188,0.9123,0.9188,0.9111
3,LinearSVC,0.8625,0.8619,0.8639,0.8619,0.8305



Mejor modelo: RandomForest

Classification report (supervised vs KMeans labels en test):
              precision    recall  f1-score   support

   Cluster 0     0.9970    0.9941    0.9956       338
   Cluster 1     1.0000    0.9663    0.9829        89
   Cluster 2     0.9980    0.9993    0.9987      1488
   Cluster 3     0.9973    0.9946    0.9959       736
   Cluster 4     0.9948    0.9969    0.9958       957
   Cluster 5     0.9904    0.9942    0.9923       520

    accuracy                         0.9961      4128
   macro avg     0.9963    0.9909    0.9935      4128
weighted avg     0.9961    0.9961    0.9961      4128



<div style="background-color: #90EE90; padding: 20px; border-radius: 10px;">

### Step 5: Save the models
Store both models in the corresponding folder.
</div

In [7]:


output_dir = Path('../models/09Kmeanshousing/')
output_dir.mkdir(parents=True, exist_ok=True)

dump(kmeans_train, open(output_dir / "k-means_housing.sav", "wb"))
dump(best_clf, open(output_dir / "random_forest.sav", "wb"))

print(f"✓ KMeans model saved to {output_dir / 'k-means_housing.sav'}")
print(f"✓ {best_name} model saved to {output_dir / 'random_forest.sav'}")

✓ KMeans model saved to ../models/09Kmeanshousing/k-means_housing.sav
✓ RandomForest model saved to ../models/09Kmeanshousing/random_forest.sav
